In [5]:
import pandas as pd

keys = ["unit_id", "year", "month"]

def read_table(path, value_col, new_name):
    """Baca CSV GEE, ambil kunci + 1 kolom nilai, ubah namanya."""
    df = pd.read_csv(path, dtype={"unit_id": str})
    df = df[keys + [value_col]].rename(columns={value_col: new_name})
    return df

# --- EVI aktual & anomali ---
# value_col: sesuaikan dengan nama kolom di CSV Anda ('mean', 'EVI', dst.)
evi = read_table("evi_table_unit_bulanan.csv", "mean", "EVI")

# Berapa baris per unit per bulan?
print(evi.groupby(keys).size().value_counts())

# Apakah duplikatnya identik (semua kolom sama)?
print("Duplikat identik penuh:", evi.duplicated(keep=False).sum())

evi_anom_raw = pd.read_csv("eviAnomaly_table_unit_bulanan_v3.csv", dtype={"unit_id": str})
evi_anom = evi_anom_raw[keys + ["lulc_dominant", "EVI_anomaly"]]

df = evi.merge(evi_anom, on=keys, how="outer", validate="1:1")

# --- Curah hujan (mm) ---
precip = read_table("chirps_table_unit_bulanan.csv", "mean", "CHIRPS (mm)")
df = df.merge(precip, on=keys, how="left", validate="1:1")

# --- SPI 1, 3, 6 ---
for s in [1, 3, 6]:
    spi = read_table(f"spi{s}_table_unit_bulanan.csv", "SPI_value", f"SPI{s}")
    df = df.merge(spi, on=keys, how="left", validate="1:1")



# --- Rapikan ---
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df[(df["year"] >= 2005) & (df["year"] <= 2025)]   # jaga-jaga jika curah hujan/SPI dari 1985
df = df.sort_values(["unit_id", "date"]).reset_index(drop=True)

df.to_csv("tnghs_all_vars.csv")

1    3024
Name: count, dtype: int64
Duplikat identik penuh: 0


In [2]:
keys = ["unit_id", "year", "month"]
files = {
    "evi":    "evi_table_unit_bulanan.csv",
    "anom":   "eviAnomaly_table_unit_bulanan_v3.csv",
    "chirps": "chirps_table_unit_bulanan.csv",
    "spi1":   "spi1_table_unit_bulanan.csv",
    "spi3":   "spi3_table_unit_bulanan.csv",
    "spi6":   "spi6_table_unit_bulanan.csv",
}

for name, path in files.items():
    d = pd.read_csv(path, dtype={"unit_id": str})
    print(f"\n[{name}] {path}")
    print("  kolom   :", d.columns.tolist())
    print("  baris   :", len(d), "| unit unik:", d["unit_id"].nunique(),
          "| tahun:", d["year"].min(), "-", d["year"].max())
    print("  duplikat kunci:", d.duplicated(keys).sum())


[evi] evi_table_unit_bulanan.csv
  kolom   : ['system:index', 'EVI_anomaly', 'date', 'frac_dominant', 'frac_hutan', 'frac_nonhutan', 'frac_pertanian', 'frac_stable', 'is_homogen', 'lulc_dominant', 'mean', 'month', 'unit_id', 'year', '.geo']
  baris   : 3024 | unit unik: 12 | tahun: 2005.0 - 2025.0
  duplikat kunci: 0

[anom] eviAnomaly_table_unit_bulanan_v3.csv
  kolom   : ['system:index', 'EVI_anomaly', 'date', 'frac_dominant', 'frac_hutan', 'frac_nonhutan', 'frac_pertanian', 'frac_stable', 'is_homogen', 'lulc_dominant', 'mean', 'month', 'unit_id', 'year', '.geo']
  baris   : 3024 | unit unik: 12 | tahun: 2005.0 - 2025.0
  duplikat kunci: 0

[chirps] chirps_table_unit_bulanan.csv
  kolom   : ['system:index', 'CHIRPS (mm)', 'date', 'frac_dominant', 'frac_hutan', 'frac_nonhutan', 'frac_pertanian', 'frac_stable', 'is_homogen', 'lulc_dominant', 'mean', 'month', 'unit_id', 'year', '.geo']
  baris   : 3024 | unit unik: 12 | tahun: 2005.0 - 2025.0
  duplikat kunci: 0

[spi1] spi1_table_unit

In [3]:
keys = ["unit_id", "year", "month"]

def read_var(path, new_name, value_col="mean"):
    d = pd.read_csv(path, dtype={"unit_id": str})
    d["year"] = d["year"].astype(int)
    d["month"] = d["month"].astype(int)
    return d[keys + [value_col]].rename(columns={value_col: new_name})

evi    = read_var("evi_table_unit_bulanan.csv", "EVI")
anom   = read_var("eviAnomaly_table_unit_bulanan_v3.csv", "EVI_anomaly")
precip = read_var("chirps_table_unit_bulanan.csv", "precip_mm")

df = evi.merge(anom, on=keys, how="outer", validate="1:1")
df = df.merge(precip, on=keys, how="left", validate="1:1")

for s in [1, 3, 6]:
    spi = read_var(f"spi{s}_table_unit_bulanan.csv", f"SPI{s}")
    df = df.merge(spi, on=keys, how="left", validate="1:1")

# Atribut statis per unit (sekali saja, bukan per bulan)
attrs = ["unit_id", "lulc_dominant", "frac_dominant", "frac_hutan",
         "frac_nonhutan", "frac_pertanian", "frac_stable", "is_homogen"]
units = (pd.read_csv("eviAnomaly_table_unit_bulanan_v3.csv", dtype={"unit_id": str})[attrs]
           .drop_duplicates("unit_id"))
df = df.merge(units, on="unit_id", how="left", validate="m:1")

df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df.sort_values(["unit_id", "date"]).reset_index(drop=True)
df.to_parquet("tnghs_all_vars.parquet")

In [4]:
print(len(df), "(seharusnya 12 x 252 = 3024)")
print(df[["EVI","EVI_anomaly","precip_mm","SPI1","SPI3","SPI6"]].isna().sum())
# Harapan sementara: SPI3 NaN = 24, SPI6 NaN = 60 (akan hilang setelah SPI diekspor ulang)

# Kelas LULC dan homogenitas tiap unit
print(units.sort_values("lulc_dominant"))

3024 (seharusnya 12 x 252 = 3024)
EVI            619
EVI_anomaly    619
precip_mm        0
SPI1             0
SPI3            24
SPI6            60
dtype: int64
       unit_id  lulc_dominant  frac_dominant  frac_hutan  frac_nonhutan  \
0   +5725+1134            1.0       0.886908    0.886908       0.037387   
1   +5726+1133            1.0       0.778835    0.778835       0.110937   
2   +5726+1135            1.0       0.944229    0.944229       0.053881   
3   +5727+1135            1.0       0.975695    0.975695       0.021505   
4   +5728+1135            1.0       0.818130    0.818130       0.143554   
5   +5729+1134            1.0       0.944752    0.944752       0.050620   
6   +5729+1137            1.0       0.609933    0.609933       0.330709   
7   +5730+1136            1.0       0.969063    0.969063       0.022297   
8   +5731+1134            1.0       0.916195    0.916195       0.075771   
9   +5731+1136            1.0       0.889641    0.889641       0.101579   
10  +5733+1135